In [ ]:
# 파일 불러오기/저장을 위한 구글 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

# ◆ 0. Actor 클러스터링 완료한 pickle 파일 불러오기

In [ ]:
import pandas as pd      # 데이터프레임 처리를 위한 pandas
import pickle            # 파이썬 객체를 파일로 저장/불러오기 위한 pickle
from tqdm import tqdm    # 반복문 진행 상황을 시각적으로 표시하는 진행바(progress bar)
import warnings
warnings.filterwarnings('ignore')  # 불필요한 경고 메시지를 출력하지 않도록 설정

# ◆ 1. LDA를 위한 데이터 전처리
* 1.1. 전체 단어의 사전 만들고 각 문서에 매칭하기
    > 단어들이 문서에 얼마나 위치해 있는지 분포를 파악해야하기 때문에 분석할 전체 데이터 대상으로
    사용된 단어의 사전을 생성해야한다,


In [ ]:
pip install gensim

In [ ]:
import gensim                          # 토픽 모델링 및 자연어 처리 라이브러리
from gensim import corpora, models     # 코퍼스(말뭉치) 및 모델 관련 모듈
from gensim.corpora import Dictionary  # 단어-ID 매핑 사전 생성 클래스 (딕셔너리 자동 생성)

### 1.1 전체 단어의 사전 만들고 각 문서에 매칭하기

# ◆ 2. LDA모델 만들기
* 2.1 LDA 기본 모델 만들기
* 2.2 LDA 토픽개수 선정
* 2.3 선정한 토픽 개수로 Action 넘버 매칭

### 2.1 LDA 기본 모델 만들기

####  하이퍼파라메터 설명

**1. num_topics : 추출할 토픽 수**
* 데이터 양이 적고 문서 내용이 단순할수록 작게(3~5) 설정합니다.
* 문서 수가 많고 주제가 다양한 경우 10~30 범위에서 후보를 두고 퍼플렉서티·토픽 일관성(coherence) 등을 비교합니다.

**2. passes : 단어수와 빈도수로 변환된 데이터 corpus를 몇번 반복해서 수렴시킬 것인지 (기본값 1)**
  * 소규모 데이터는 10~20 정도에서 시작하며, 데이터 크기가 크고 토픽이 잘 안 나오는 경우 점진적으로 증가합니다.
  * 너무 크면 시간 증가 + 과적합 위험이 있습니다.

**3. iterations : 전체 문서 반복 횟수 (기본값 50)**
* 일반적으로 30~100 사이에서 사용합니다.
* 문서가 짧거나, 데이터가 적을 경우 iterations 값을 늘려야 수렴이 잘 됩니다.
* 만약 passes를 늘렸다면 iterations은 과하게 키우지 않고 균형 유지하는 것이 좋습니다.

4. random_state : 실험을 반복하며 비교해야 할 때는 고정된 정수(예: 0, 42) 사용합니다.
  * 만약 매번 다른 토픽을 일부러 보고 싶다면 생략 가능합니다.

* 현재 설정(num_topics=3, passes=20, iterations=50)은 클러스터 안의 문서 수가 많지 않고, 비교적 “굵은” 주제 3개를 뽑겠다는 가정에서는 무난한 시작값에 해당합니다.

### 2.2 LDA 토픽 수 선정
- Perplexity(혼잡도) : 낮을수록 모델이 문서를 잘 예측함
- Coherence(일관성)  : 높을수록 토픽 내 단어들이 의미적으로 연관성이 높음

In [ ]:
from gensim.models import CoherenceModel  # Coherence 점수 계산 클래스
import matplotlib.pyplot as plt           # 그래프 시각화
import numpy as np                        # 수치 연산

> 2.2.1 Perplexity&Coherence 점수 계산해보기

> 2.2.1 Perplexity&Cohearence 그래프 만들기

### 2.3 선정한 토픽 개수로 Action 넘버 매칭

# ◆ 3. LDA 시각화 (LDAvis)
* LDAvis : 각 토픽의 크기·거리·주요 단어를 인터랙티브하게 시각화하는 도구입니다.
  - 원의 크기  : 해당 토픽이 전체 문서에서 차지하는 비중
  - 원의 거리  : 토픽 간 유사도 (가까울수록 비슷한 주제)
  - 오른쪽 막대: 토픽 내 단어의 중요도

# ◆ 4. LDA 분석을 위한 후작업
* LDAvis와 LDA모델의 토픽 넘버 매칭
* LDAvis의 토픽 번호(1,2,3,4)와 LDA 모델의 토픽 번호(0,1,2,3)는 서로 다른 순서로 정렬될 수 있으므로 수동으로 대응 관계를 확인해야 합니다.